# Nemotron LoRA — train from Colab via Modal (A100-80GB, bf16)

**Why not Colab's own GPU?** Colab's A100 is 40 GB, which forces 8-bit, and 8-bit on this model's custom MoE hits a dtype error (bitsandbytes fp16 vs bf16 stream). The working path is **bf16 on an 80 GB A100**, which Colab doesn't offer but **Modal** does. So we use Colab only to *launch* the Modal job — set the runtime to **CPU** (Runtime → Change runtime type → CPU); no Colab GPU is needed.

The Modal job is fully self-contained: it pulls the data with your Kaggle token, builds the SFT set, trains, and saves the adapter to a Modal Volume.

## 1. Install Modal + authenticate
`modal setup` prints a link — open it, approve, come back. One time.

In [ ]:
!pip install -q modal
!modal setup

## 2. Store your Kaggle token as a Modal secret
Replace the placeholder with your real KGAT token. (Add `HF_TOKEN=hf_...` too only if the base-model download 401s.)

In [ ]:
!modal secret create nemotron KAGGLE_TOKEN=KGAT_xxxxxxxxxxxxxxxx

## 3. Get the Modal script and launch training (detached)
`--detach` keeps the job running on Modal even if Colab disconnects. The build compiles nothing (prebuilt torch 2.7 + mamba wheels); first run downloads the ~63 GB model (cached afterwards), then trains 2 epochs in bf16. ~1-3 h.

In [ ]:
!rm -rf repo && git clone -b build/nemotron-pipeline https://github.com/SebAustin/NVIDIA-Nemotron-Model-Reasoning-Challenge repo
!cd repo && modal run --detach modal/train_modal.py

## 4. Download the trained adapter
Run after the job finishes (watch it at modal.com -> app `nemotron-lora`).

In [ ]:
!cd repo && modal volume get nemotron-out lora_adapter ./lora_adapter
import shutil; shutil.make_archive('/content/lora_adapter','zip','repo/lora_adapter')
from google.colab import files; files.download('/content/lora_adapter.zip')

## 5. Next: package + submit on Kaggle
Upload `lora_adapter` as a Kaggle dataset, then run `kaggle_package_submit.ipynb` in a Kaggle notebook -> Save Version -> Submit.

---

**Watching progress:** the Modal dashboard streams logs. Look for the per-step loss and, at the end, the per-category min-logprob (which families to upweight next).